<a href="https://colab.research.google.com/github/sakuna47/real-time-multilingual-pii-detection/blob/Data/02_explore_ai4privacy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
from collections import Counter

BASE = "/content/drive/MyDrive/PII_Project"
train = pd.read_parquet(f"{BASE}/data/raw/ai4privacy/train.parquet")
val   = pd.read_parquet(f"{BASE}/data/raw/ai4privacy/validation.parquet")
print("Train rows:", len(train), "| Validation rows:", len(val))

Mounted at /content/drive
Train rows: 177677 | Validation rows: 47728


In [2]:
label_counts = Counter()
for spans in train["privacy_mask"]:
    for s in spans:
        label_counts[s["label"]] += 1

print("Number of different labels:", len(label_counts))
for label, n in label_counts.most_common():
    print(f"{label:25s} {n}")

Number of different labels: 28
TIME                      89225
USERNAME                  65405
IDCARD                    60014
SOCIALNUMBER              59142
EMAIL                     58655
PASSPORT                  56759
DRIVERLICENSE             55758
BOD                       54701
LASTNAME1                 54272
IP                        51724
GIVENNAME1                48494
SEX                       47606
CITY                      46460
TEL                       46410
POSTCODE                  45316
STREET                    44645
STATE                     44557
BUILDING                  44454
TITLE                     40751
COUNTRY                   40040
DATE                      39735
PASS                      38296
SECADDRESS                18542
LASTNAME2                 13634
GIVENNAME2                12796
GEOCOORD                  4557
LASTNAME3                 4393
CARDISSUER                50


In [3]:
examples = {}
for spans in train["privacy_mask"]:
    for s in spans:
        if s["label"] not in examples:
            examples[s["label"]] = s["value"]
for label, value in sorted(examples.items()):
    print(f"{label:25s} → {value}")

BOD                       → Feb 8, 1986
BUILDING                  → 163
CARDISSUER                → Diners Club International
CITY                      → Bristol
COUNTRY                   → GB
DATE                      → August 5th, 2057
DRIVERLICENSE             → LOUMA.657200.9.504
EMAIL                     → bballoi@yahoo.com
GEOCOORD                  → [53.2, -1.3474]
GIVENNAME1                → Mayza
GIVENNAME2                → Wongduean
IDCARD                    → KB90324ER
IP                        → 7836:3dcf:9edf:692:fd5f:4de5:a9d6:da24
LASTNAME1                 → Balloi
LASTNAME2                 → Eckrich
LASTNAME3                 → Raizner
PASS                      → q4R\\
PASSPORT                  → 301025226
POSTCODE                  → BS34 7HU, BS34 7HZ
SECADDRESS                → PB 73
SEX                       → M
SOCIALNUMBER              → 996 076 6460
STATE                     → ENG
STREET                    → Conygre Grove
TEL                       → +1322380 4181
T

In [4]:
lengths = train["source_text"].str.len()
print(lengths.describe())

count    177677.000000
mean        420.976514
std          57.002062
min          36.000000
25%         387.000000
50%         426.000000
75%         464.000000
max         512.000000
Name: source_text, dtype: float64


In [5]:
import os, json
from collections import Counter

MAP = {
    "GIVENNAME1": "PERSON", "GIVENNAME2": "PERSON",
    "LASTNAME1": "PERSON", "LASTNAME2": "PERSON", "LASTNAME3": "PERSON",
    "IDCARD": "ID_NUMBER", "SOCIALNUMBER": "ID_NUMBER",
    "PASSPORT": "ID_NUMBER", "DRIVERLICENSE": "ID_NUMBER",
    "TEL": "PHONE", "EMAIL": "EMAIL",
    "BUILDING": "ADDRESS", "STREET": "ADDRESS",
    "SECADDRESS": "ADDRESS", "POSTCODE": "ADDRESS",
    "CITY": "LOCATION", "GEOCOORD": "LOCATION",
    "BOD": "DOB", "USERNAME": "USERNAME", "PASS": "PASSWORD", "IP": "IP_ADDRESS",
    # dropped → O: TIME, DATE, SEX, TITLE, STATE, COUNTRY, CARDISSUER
}

def convert(text, spans):
    out = []
    for s in sorted(spans, key=lambda x: x["start"]):
        label = MAP.get(s["label"])
        if label is None:
            continue
        if out and out[-1]["label"] == label:
            gap = text[out[-1]["end"]:s["start"]]
            if (label == "PERSON" and gap.strip() == "") or \
               (label == "ADDRESS" and gap.strip(" ,") == ""):
                out[-1]["end"] = int(s["end"])
                continue
        out.append({"start": int(s["start"]), "end": int(s["end"]), "label": label})
    return out

def convert_split(df, name):
    out_dir = f"{BASE}/data/processed/ai4privacy"
    os.makedirs(out_dir, exist_ok=True)
    path = f"{out_dir}/{name}.jsonl"
    bad = 0
    with open(path, "w", encoding="utf-8") as f:
        for _, row in df.iterrows():
            text = row["source_text"]
            if any(text[s["start"]:s["end"]] != s["value"] for s in row["privacy_mask"]):
                bad += 1
                continue
            rec = {"id": row["id"], "text": text,
                   "spans": convert(text, row["privacy_mask"]),
                   "lang": row["language"], "source": "ai4privacy"}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"{name}: saved {len(df) - bad} rows, skipped {bad} → {path}")

convert_split(train, "train")
convert_split(val, "validation")

train: saved 176495 rows, skipped 1182 → /content/drive/MyDrive/PII_Project/data/processed/ai4privacy/train.jsonl
validation: saved 47414 rows, skipped 314 → /content/drive/MyDrive/PII_Project/data/processed/ai4privacy/validation.jsonl


In [6]:
new_counts = Counter()
first = None
with open(f"{BASE}/data/processed/ai4privacy/train.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        first = first or rec
        for s in rec["spans"]:
            new_counts[s["label"]] += 1
t$0
for label, n in new_counts.most_common():
    print(f"{label:12s} {n}")

print("\nExample:")
for s in first["spans"]:
    print(f'{s["label"]:10s} → {first["text"][s["start"]:s["end"]]}')

ID_NUMBER    229467
ADDRESS      137125
PERSON       117916
USERNAME     64618
EMAIL        58187
DOB          54229
IP_ADDRESS   51243
LOCATION     50594
PHONE        45972
PASSWORD     36911

Example:
USERNAME   → wynqvrh053
USERNAME   → luka.burg
USERNAME   → qahil.wittauer
USERNAME   → gholamhossein.ruschke
USERNAME   → pdmjrsyoz1460


In [7]:
shown = 0
with open(f"{BASE}/data/processed/ai4privacy/train.jsonl", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        labels = {s["label"] for s in rec["spans"]}
        if "PERSON" in labels and "ADDRESS" in labels:
            for s in rec["spans"]:
                if s["label"] in ("PERSON", "ADDRESS"):
                    print(f'{s["label"]:8s} → {rec["text"][s["start"]:s["end"]]}')
            print("-" * 40)
            shown += 1
            if shown == 3:
                break

ADDRESS  → 163
ADDRESS  → Conygre Grove
ADDRESS  → BS34 7HU, BS34 7HZ
PERSON   → Baasgaran Palmoso
ADDRESS  → 646
ADDRESS  → School Lane
----------------------------------------
PERSON   → Helbert
ADDRESS  → 53, Zantelweg
ADDRESS  → 3161
----------------------------------------
PERSON   → Guirard
ADDRESS  → 397, Kostverlorenkade
ADDRESS  → 1183 TM
ADDRESS  → PB 73
PERSON   → Potkonjak
PERSON   → Ucha
PERSON   → Mastrogiacomo
PERSON   → Raizner
----------------------------------------
